[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_07/MFM_ch7_seminar/MFM_ch7_seminar.ipynb)

# MFM_ch7_seminar

Seminar 7 computations: Normal and Student-t VaR/ES step by step, discrete losses and the failure of subadditivity, component VaR, Cornish-Fisher, historical VaR/ES on BET with bootstrap intervals, GPD fit and EVT-VaR on BET, filtered vs historical simulation on Bitcoin, Cornish-Fisher vs empirical quantiles, Euler contributions for a four-asset portfolio, i.i.d. and block bootstrap intervals for ES, and the ES 2.5% capital of a BVB blue-chip portfolio versus an S&P 500 portfolio.

*Modelling Financial Markets (MFM), Chapter 7: Value-at-Risk and Expected Shortfall. Author: Daniel Traian Pele.*

In [ ]:
# Colab: !pip install arch
import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, integrate
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'   # doar linii de referinta, benzi, grila
LightGray = '#DADADA'
Teal     = '#17A2B8'
METHOD_COL = {'HS': MainBlue, 'Normal': Orange, 'Student-t': Forest, 'Cornish-Fisher': Purple, 'EVT (GPD)': IDAred}

HERE = '.'
SEED = 42
ORDER = ['sp500', 'bet', 'btc', 'eurron', 'gold']
METHODS = ['HS', 'Normal', 'Student-t', 'Cornish-Fisher', 'EVT (GPD)']
B_BOOT = 2000


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# nume -> (simbol, eticheta, tip, data de start)
MARKETS = {
    'sp500':  ('GSPC.INDX',    'S&P 500',                  'index',  '2000-01-01'),
    'bet':    ('BET',          'BET',                      'index',  '2000-01-01'),
    'btc':    ('BTC-USD.CC',   'Bitcoin',                  'crypto', '2014-09-17'),
    'eurron': ('REF:EUR',      'EUR/RON (reference rate)', 'fx',     '2005-07-01'),
    'gold':   ('XAUUSD.FOREX', 'Gold (XAU/USD)',           'fx',     '2000-01-01'),
}
# active pentru portofolii (join pe preturi in zilele comune)
ASSETS = {
    'SPY': ('SPY.US', 'adjusted_close'), 'TLT': ('TLT.US', 'adjusted_close'), 'GLD': ('GLD.US', 'adjusted_close'),
    'BTC': ('BTC-USD.CC', 'close'),
    'TLV': ('TLV.RO', 'adjusted_close'), 'SNP': ('SNP.RO', 'adjusted_close'), 'BRD': ('BRD.RO', 'adjusted_close'),
    'TGN': ('TGN.RO', 'adjusted_close'), 'SNG': ('SNG.RO', 'adjusted_close'), 'SNN': ('SNN.RO', 'adjusted_close'),
    'EL': ('EL.RO', 'adjusted_close'), 'FP': ('FP.RO', 'adjusted_close'), 'TEL': ('TEL.RO', 'adjusted_close'),
}
LABELS = {k: v[1] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Cursul oficial de referinta RON (arhive XML anuale BNR)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Pretul zilnic, curatat dupa conventiile capitolului."""
    symbol, _, kind, start0 = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    s = read_market(symbol)['close'].loc[start:end]
    s = s[s > 0].dropna()
    if kind != 'crypto':
        s = s[s.index.dayofweek < 5]          # fara cotatii de weekend
    if kind == 'index':
        s = s[s.diff() != 0]                  # fara sarbatori completate cu pretul anterior
    return s.rename(name)


def log_returns(name, start=None, end=END):
    """Randamente log pe calendarul propriu al seriei."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


def periods_per_year(r):
    """Frecventa reala: numarul mediu de observatii pe an calendaristic."""
    return len(r) / ((r.index[-1] - r.index[0]).days / 365.25)


def asset_price(key, end=END):
    symbol, col = ASSETS[key]
    s = read_market(symbol)[col].loc[:end]
    s = s[s > 0].dropna()
    if key != 'BTC':
        s = s[s.index.dayofweek < 5]
    return s.rename(key)


def joint_returns(keys, start=None, end=END):
    """Randamente log simple pentru mai multe active: join pe PRETURI in zilele comune, apoi randamente."""
    p = pd.concat([asset_price(k, end) for k in keys], axis=1, join='inner').dropna()
    if start:
        p = p.loc[start:]
    return np.log(p).diff().dropna()


# randamente log zilnice in %, pierderea L = -r
rets = {k: 100 * log_returns(k) for k in MARKETS}
loss = {k: -v for k, v in rets.items()}
for k in ORDER:
    print(f'{LABELS[k]:26s}', rets[k].index[0].date(), '->', rets[k].index[-1].date(), len(rets[k]), 'obs')

## Risk measures

In [ ]:
def hs_var_es(L, alpha):
    """Simulare istorica: VaR alpha = cuantila empirica 1 - alpha a pierderii; ES = media pierderilor de dincolo."""
    L = np.asarray(L)
    v = np.quantile(L, 1 - alpha)
    return v, L[L >= v].mean()


def normal_var_es(L, alpha):
    """Distributia Normala: VaR = -mu_X + sigma z_{1-alpha}, ES = -mu_X + sigma phi(z_alpha)/alpha (mu_L = -mu_X)."""
    mu, s = np.mean(L), np.std(L, ddof=1)
    z = stats.norm.ppf(1 - alpha)
    return mu + s * z, mu + s * stats.norm.pdf(z) / alpha


def t_fit(L):
    """Student-t cu locatie si scala, estimata prin verosimilitate maxima."""
    return stats.t.fit(np.asarray(L))


def t_var_es(L, alpha, params=None):
    """Student-t pe pierderi: VaR = m + s t_{1-alpha}, ES = m + s g(t_alpha)/alpha (nu + t_alpha^2)/(nu - 1)."""
    nu, m, s = params if params is not None else t_fit(L)
    q = stats.t.ppf(1 - alpha, nu)
    return m + s * q, m + s * stats.t.pdf(q, nu) / alpha * (nu + q ** 2) / (nu - 1)


def cf_quantile(z, S, K):
    """Cuantila Cornish-Fisher standardizata: z = cuantila Normala, S = asimetria, K = excesul de aplatizare."""
    return z + (z ** 2 - 1) * S / 6 + (z ** 3 - 3 * z) * K / 24 - (2 * z ** 3 - 5 * z) * S ** 2 / 36


def cf_var_es(L, alpha):
    """Cornish-Fisher pe randamente X = -L: VaR = -(mu_X + sigma z~_alpha); ES = media lui VaR_u pentru u in (0, alpha)."""
    X = -np.asarray(L)
    mu, s = X.mean(), X.std(ddof=1)
    S, K = stats.skew(X), stats.kurtosis(X)
    var = -(mu + s * cf_quantile(stats.norm.ppf(alpha), S, K))
    es = -integrate.quad(lambda u: mu + s * cf_quantile(stats.norm.ppf(u), S, K), 0, alpha, limit=200)[0] / alpha
    return var, es


def gpd_fit(L, tail=0.05):
    """POT: pragul u lasa deasupra proportia tail din pierderi; GPD estimata prin verosimilitate maxima pe excese."""
    L = np.asarray(L)
    u = np.quantile(L, 1 - tail)
    ex = L[L > u] - u
    xi, _, beta = stats.genpareto.fit(ex, floc=0)
    return dict(u=u, xi=xi, beta=beta, nu=len(ex), n=len(L))


def gpd_var_es(fit, alpha):
    """VaR si ES cu probabilitatea cozii alpha din coada GPD (estimatorul de tip Smith)."""
    u, xi, beta, nu, n = fit['u'], fit['xi'], fit['beta'], fit['nu'], fit['n']
    var = u + beta / xi * (((n / nu) * alpha) ** (-xi) - 1)
    return var, var / (1 - xi) + (beta - xi * u) / (1 - xi)


def gpd_se(fit):
    """Erori standard asimptotice ale MLE pentru GPD (xi > -1/2)."""
    xi, beta, nu = fit['xi'], fit['beta'], fit['nu']
    return np.sqrt((1 + xi) ** 2 / nu), np.sqrt(2 * beta ** 2 * (1 + xi) / nu)


def mean_excess(L, us):
    """Functia de exces mediu e(u) = E[L - u | L > u] pe o grila de praguri."""
    L = np.asarray(L)
    return np.array([(L[L > u] - u).mean() for u in us]), np.array([(L > u).sum() for u in us])


def garch_filter(r, params=None, last_obs=None):
    """AR(1)-GARCH(1,1) cu QML Normal (Capitolul 5); r in %.
    Intoarce modelul estimat (pana la last_obs) si seriile mu_t, sigma_t pentru toata selectia."""
    am = arch_model(r, mean='AR', lags=1, vol='GARCH', p=1, q=1, dist='normal', rescale=False)
    if params is None:
        params = am.fit(disp='off', last_obs=last_obs).params
    fx = am.fix(params)
    sig = fx.conditional_volatility
    mu = r - fx.resid
    return params, mu, sig


def garch_next(r, params):
    """Media si volatilitatea conditionate pentru ziua urmatoare ultimei observatii."""
    c, phi, om, al, be = params.values[:5]
    _, mu, sig = garch_filter(r, params)
    e = r.iloc[-1] - mu.iloc[-1]
    return c + phi * r.iloc[-1], np.sqrt(om + al * e ** 2 + be * sig.iloc[-1] ** 2)


def fhs_mc(r, params, z, h, n_paths=100_000, seed=42):
    """Monte Carlo FHS: traiectorii AR(1)-GARCH(1,1) pe h zile cu reziduuri standardizate reesantionate.
    Intoarce pierderile cumulate pe h zile (in %, randamente log)."""
    rng = np.random.default_rng(seed)
    c, phi, om, al, be = params.values[:5]
    _, mu, sig = garch_filter(r, params)
    r_prev = np.full(n_paths, r.iloc[-1])
    e_prev = np.full(n_paths, r.iloc[-1] - mu.iloc[-1])
    s2_prev = np.full(n_paths, sig.iloc[-1] ** 2)
    tot = np.zeros(n_paths)
    for _ in range(h):
        s2 = om + al * e_prev ** 2 + be * s2_prev
        e = np.sqrt(s2) * rng.choice(z, n_paths)
        r_new = c + phi * r_prev + e
        tot += r_new
        r_prev, e_prev, s2_prev = r_new, e, s2
    return -tot


def rolling_conditional(r, first_year, window_hs=500, alpha=0.01, tail_evt=0.10):
    """VaR/ES conditionat zi de zi: parametrii AR(1)-GARCH re-estimati la fiecare inceput de an
    pe toate datele anterioare; FHS si EVT conditionat folosesc reziduurile standardizate anterioare.
    Pentru comparatie: HS pe fereastra mobila de window_hs zile. Fara informatie din viitor."""
    out = []
    L = -r
    for y in range(first_year, r.index[-1].year + 1):
        est = r.loc[:f'{y - 1}-12-31']
        params, mu, sig = garch_filter(r, last_obs=f'{y}-01-01')
        z = ((est - mu.loc[est.index]) / sig.loc[est.index]).dropna()
        nz = -z.values
        fz = gpd_fit(nz, tail_evt)
        q_fhs, q_evt_v = np.quantile(nz, 1 - alpha), gpd_var_es(fz, alpha)[0]
        es_fhs = nz[nz >= q_fhs].mean()
        es_evt = gpd_var_es(fz, alpha)[1]
        days = r.loc[f'{y}-01-01':f'{y}-12-31'].index
        for d in days:
            i = r.index.get_loc(d)
            past = L.iloc[max(0, i - window_hs):i]
            out.append((d, L.loc[d], -mu.loc[d] + sig.loc[d] * q_fhs, -mu.loc[d] + sig.loc[d] * es_fhs,
                        -mu.loc[d] + sig.loc[d] * q_evt_v, -mu.loc[d] + sig.loc[d] * es_evt,
                        -mu.loc[d] + sig.loc[d] * stats.norm.ppf(1 - alpha), np.quantile(past, 1 - alpha), sig.loc[d]))
    return pd.DataFrame(out, columns=['date', 'loss', 'fhs_var', 'fhs_es', 'cevt_var', 'cevt_es', 'ngarch_var',
                                      'hs_var', 'sigma']).set_index('date')

## Helper functions

In [ ]:
BVB = ['TLV', 'SNP', 'BRD', 'TGN', 'SNG', 'SNN', 'EL', 'TEL']

C_START = '2016-09-19'

def discrete_var_es(values, probs, alpha):
    """VaR si ES cu probabilitatea cozii alpha pentru o pierdere discreta L = -X (formula Acerbi-Tasche).
    VaR_alpha = -inf{x : P(X <= x) > alpha}; ES_alpha = (E[L 1{L > VaR}] + VaR (alpha - P(L > VaR))) / alpha."""
    order = np.argsort(values)
    x, p = np.asarray(values)[order], np.asarray(probs)[order]
    cdf = np.cumsum(p)
    var = x[np.searchsorted(cdf, 1 - alpha - 1e-12)]
    tail = x > var
    es = ((x[tail] * p[tail]).sum() + var * (alpha - p[tail].sum())) / alpha
    return var, es


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o

## Analysis

In [ ]:
def a1_normal(mu=0.04, sigma=1.2, position=1_000_000):
    """VaR si ES pentru distributia Normala, pas cu pas: X ~ N(mu, sigma^2) (randamente in %),
    VaR_alpha = -(mu + sigma z_alpha), ES_alpha = -mu + sigma phi(z_alpha) / alpha."""
    out = dict(mu=mu, sigma=sigma, position=position)
    for a, tag in [(0.01, '1'), (0.025, '2_5')]:
        z = stats.norm.ppf(a)                  # z_alpha < 0
        phi = stats.norm.pdf(z)
        out[f'z{tag}'] = z
        out[f'phi{tag}'] = phi
        out[f'k{tag}'] = phi / a
        out[f'var{tag}'] = -(mu + sigma * z)
        out[f'es{tag}'] = -mu + sigma * phi / a
    out['var1_10'] = -np.sqrt(10) * sigma * out['z1']
    return out


def a2_student(nu=4, mu=0.04, sigma=1.2):
    """Student-t standardizata la aceeasi dispersie: X = mu + s T, s = sigma sqrt((nu-2)/nu);
    VaR_alpha = -(mu + s t_alpha), ES_alpha = -mu + s g(t_alpha)/alpha (nu + t_alpha^2)/(nu - 1)."""
    s = sigma * np.sqrt((nu - 2) / nu)
    out = dict(nu=nu, s=s)
    for a, tag in [(0.01, '1'), (0.025, '2_5')]:
        q = stats.t.ppf(a, nu)                 # t_alpha < 0
        g = stats.t.pdf(q, nu)
        k = g / a * (nu + q ** 2) / (nu - 1)
        out[f'q{tag}'] = q
        out[f'g{tag}'] = g
        out[f'k{tag}'] = k
        out[f'var{tag}'] = -(mu + s * q)
        out[f'es{tag}'] = -mu + s * k
    return out


def a3_bond(p=0.03, lgd=60.0):
    out = dict(p=p, lgd=lgd)
    for a, tag in [(0.05, '5'), (0.025, '2_5')]:
        v, e = discrete_var_es([0, lgd], [1 - p, p], a)
        out[f'var{tag}'] = v
        out[f'es{tag}'] = e
    return out


def a4_bonds(p=0.007, lgd=100.0):
    out = dict(p=p, lgd=lgd, p_any=1 - (1 - p) ** 2, p_both=p ** 2, p_one=2 * p * (1 - p))
    for a, tag in [(0.01, '1'), (0.015, '1_5')]:
        v1, e1 = discrete_var_es([0, lgd], [1 - p, p], a)
        vp, ep = discrete_var_es([0, lgd, 2 * lgd], [(1 - p) ** 2, 2 * p * (1 - p), p ** 2], a)
        out.update({f'varA_{tag}': v1, f'esA_{tag}': e1, f'varP_{tag}': vp, f'esP_{tag}': ep})
    return out


def a5_components(pos=(600_000, 400_000), sig=(1.2, 0.9), rho=-0.2, alpha=0.01):
    """VaR 1% pe componente pentru doua active, distributia Normala, media zero."""
    w = np.array(pos, float)
    s = np.array(sig) / 100
    S = np.array([[s[0] ** 2, rho * s[0] * s[1]], [rho * s[0] * s[1], s[1] ** 2]])
    sp = np.sqrt(w @ S @ w)
    z = stats.norm.ppf(1 - alpha)              # z_{1-alpha} = -z_alpha > 0
    Sw = S @ w
    mvar = z * Sw / sp
    cvar = w * mvar
    return dict(w=w, s=s, rho=rho, z=z, var_p_amt=z * sp, sigma_p_amt=sp, sp2=sp ** 2, Sw=Sw, mvar=mvar, cvar=cvar,
                share=cvar / (z * sp), var_ind=z * s * w, undiv=(z * s * w).sum())


def a6_cf(S=-0.5, K=3.0, sigma=1.2, mu=0.0, alpha=0.01, K2=10.0):
    """Cornish-Fisher pe randamente: asimetria S, excesul de aplatizare K; VaR_alpha = -(mu + sigma z~_alpha)."""
    z = stats.norm.ppf(alpha)                  # z_alpha < 0
    t1, t2, t3 = (z ** 2 - 1) * S / 6, (z ** 3 - 3 * z) * K / 24, -(2 * z ** 3 - 5 * z) * S ** 2 / 36
    zc = cf_quantile(z, S, K)
    zc2 = cf_quantile(z, S, K2)
    return dict(S=S, K=K, K2=K2, z=z, t1=t1, t2=t2, t3=t3, zcf=zc, var_cf=-(mu + sigma * zc), var_n=-(mu + sigma * z),
                zcf2=zc2, var_cf2=-(mu + sigma * zc2))


def boot_ci(L, fun, B=B_BOOT, block=None, seed=SEED):
    """Interval bootstrap percentil 95%: i.i.d. (block=None) sau pe blocuri mobile de lungime block."""
    rng = np.random.default_rng(seed)
    L = np.asarray(L)
    n = len(L)
    stats_ = np.empty(B)
    for b in range(B):
        if block is None:
            x = L[rng.integers(0, n, n)]
        else:
            starts = rng.integers(0, n - block + 1, int(np.ceil(n / block)))
            x = np.concatenate([L[s:s + block] for s in starts])[:n]
        stats_[b] = fun(x)
    return np.percentile(stats_, [2.5, 97.5]), stats_


def b1_bet_hs():
    L = -100 * log_returns('bet')
    out = {}
    boots = {}
    for tag, x in [('full', L), ('w500', L.iloc[-500:])]:
        v1, e1 = hs_var_es(x, 0.01)
        v2, e2 = hs_var_es(x, 0.025)
        ci, bs = boot_ci(x.values, lambda y: hs_var_es(y, 0.01)[0])
        boots[tag] = bs
        out[tag] = dict(N=len(x), start=str(x.index[0].date()), var1=v1, es1=e1, var2_5=v2, es2_5=e2,
                        ci_lo=ci[0], ci_hi=ci[1], n_beyond=int((x >= v1).sum()))
    # graficul: coada pierderilor + distributiile bootstrap
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
    ax = axes[0]
    ax.hist(L[L > 1.5], bins=np.linspace(1.5, 13.5, 61), color=MainBlue, alpha=0.55, label='BET daily losses above 1.5%')
    ax.axvline(out['full']['var1'], color=IDAred, lw=1.3, label=f"VaR 1% = {out['full']['var1']:.2f}%")
    ax.axvline(out['full']['es1'], color=IDAred, ls='--', lw=1.3, label=f"ES 1% = {out['full']['es1']:.2f}%")
    ax.set_xlabel('Daily loss (%)')
    ax.set_ylabel('Number of days')
    ax.set_title('BET, 2000-2026: historical simulation')
    legend_outside_bottom(ax, 1, -0.2)
    ax = axes[1]
    ax.hist(boots['full'], bins=40, density=True, color=MainBlue, alpha=0.55, label='Full sample (2000-2026)')
    ax.hist(boots['w500'], bins=40, density=True, color=Amber, alpha=0.55, label='Last 500 days')
    ax.set_xlabel('Bootstrap VaR 1% (%)')
    ax.set_ylabel('Density')
    ax.set_title('Sampling uncertainty of HS VaR 1%')
    legend_outside_bottom(ax, 2, -0.2)
    plt.tight_layout()
    save_fig('ch7_sem_bet_hs')
    return out


def b2_bet_gpd():
    L = (-100 * log_returns('bet')).values
    f = gpd_fit(L, 0.05)
    se_xi, se_beta = gpd_se(f)
    out = dict(u=f['u'], xi=f['xi'], beta=f['beta'], nu=f['nu'], n=f['n'], se_xi=se_xi, se_beta=se_beta)
    for a, tag in [(0.01, '1'), (0.005, '0_5'), (0.001, '0_1')]:
        v, e = gpd_var_es(f, a)
        hv, he = hs_var_es(L, a)
        out.update({f'evt_var{tag}': v, f'evt_es{tag}': e, f'hs_var{tag}': hv, f'hs_es{tag}': he,
                    f'n_beyond{tag}': int((L >= hv).sum())})
    # sensibilitatea la prag
    qs = np.linspace(0.10, 0.015, 18)          # proportia pierderilor peste prag
    xis, lo, hi, v01 = [], [], [], []
    for q in qs:
        g = gpd_fit(L, q)
        s = gpd_se(g)[0]
        xis.append(g['xi'])
        lo.append(g['xi'] - 1.96 * s)
        hi.append(g['xi'] + 1.96 * s)
        v01.append(gpd_var_es(g, 0.001)[0])
    out['xi_range'] = [float(min(xis)), float(max(xis))]
    out['v0_1_range'] = [float(min(v01)), float(max(v01))]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
    ax = axes[0]
    xs = np.sort(L[L > f['u']])
    emp = (1 - np.arange(len(xs)) / len(xs)) * f['nu'] / f['n']
    ax.loglog(xs, emp, 'o', ms=2.5, color=MainBlue, label='Empirical P(L > x)')
    xx = np.linspace(f['u'], xs[-1] * 1.3, 200)
    ax.loglog(xx, f['nu'] / f['n'] * stats.genpareto.sf(xx - f['u'], f['xi'], scale=f['beta']), color=IDAred,
              label=f"GPD fit, xi = {f['xi']:.2f}")
    ax.loglog(xx, stats.norm.sf(xx, L.mean(), L.std()), color=Orange, ls='--', label='Normal distribution')
    ax.set_ylim(1e-5, 0.08)
    ax.set_xlabel('Daily loss x (%)')
    ax.set_ylabel('Exceedance probability')
    ax.set_title('BET: largest 5% of daily losses')
    legend_outside_bottom(ax, 2, -0.2)
    ax = axes[1]
    ax.fill_between(100 * qs, lo, hi, color=LightGray, label='95% confidence band')
    ax.plot(100 * qs, xis, 'o-', ms=3, color=IDAred, label='Estimated xi')
    ax.axhline(0, color=Gray, lw=0.6)
    ax.set_xlabel('Share of losses above the threshold (%)')
    ax.set_ylabel('Shape parameter xi')
    ax.set_title('Stability of xi across thresholds')
    legend_outside_bottom(ax, 2, -0.2)
    plt.tight_layout()
    save_fig('ch7_sem_bet_gpd')
    return out


def b3_btc_fhs():
    r = 100 * log_returns('btc')
    params, mu, sig = garch_filter(r)
    z = ((r - mu) / sig).dropna().values
    m1, s1 = garch_next(r, params)
    nz = -z
    q = np.quantile(nz, 1 - 0.01)
    L = -r
    v500, e500 = hs_var_es(L.iloc[-500:], 0.01)
    vfull, efull = hs_var_es(L, 0.01)
    roll = pd.read_csv(os.path.join(HERE, 'ch7_rolling_btc.csv'), index_col=0, parse_dates=True)
    rec = roll.loc['2024-09-18':]
    out = dict(mu=m1, sigma=s1, sigma_uncond=r.std(), zq=q, zes=nz[nz >= q].mean(), fhs_var=-m1 + s1 * q,
               fhs_es=-m1 + s1 * nz[nz >= q].mean(), hs500_var=v500, hs500_es=e500, hsfull_var=vfull, hsfull_es=efull,
               n_rec=len(rec), exc_hs_rec=int((rec['loss'] > rec['hs_var']).sum()),
               exc_fhs_rec=int((rec['loss'] > rec['fhs_var']).sum()),
               exc_hs_all=float((roll['loss'] > roll['hs_var']).mean()),
               exc_fhs_all=float((roll['loss'] > roll['fhs_var']).mean()), n_all=len(roll),
               alpha=params.iloc[3], beta=params.iloc[4], sd500=r.iloc[-500:].std(), last_r=r.iloc[-1])
    fig, ax = plt.subplots(figsize=(10, 3.3))
    ax.bar(rec.index, rec['loss'].clip(lower=0), width=1, color=Teal, alpha=0.45, label='Daily loss (gains set to 0)')
    ax.plot(rec.index, rec['hs_var'], color=MainBlue, label='HS VaR 1% (500 days)')
    ax.plot(rec.index, rec['fhs_var'], color=IDAred, lw=0.9, label='FHS VaR 1%')
    ax.set_ylabel('Daily loss (%)')
    ax.set_title('Bitcoin, Sep 2024 - Sep 2026')
    legend_outside_bottom(ax, 3, -0.14)
    save_fig('ch7_sem_btc_fhs')
    return out


def b4_cf_vs_empirical():
    t = pd.read_csv(os.path.join(HERE, 'ch7_var_es_table.csv'), index_col=0)
    out = {}
    for k in t.index:
        L = (-100 * log_returns(k)).values
        ci, _ = boot_ci(L, lambda y: hs_var_es(y, 0.01)[0], B=1000)
        out[k] = dict(skew=-t.loc[k, 'skew'], kurt=t.loc[k, 'kurt'], hs1=t.loc[k, 'HS|VaR 1%'],
                      cf1=t.loc[k, 'Cornish-Fisher|VaR 1%'], n1=t.loc[k, 'Normal|VaR 1%'],
                      hs2_5=t.loc[k, 'HS|VaR 2.5%'], cf2_5=t.loc[k, 'Cornish-Fisher|VaR 2.5%'],
                      n2_5=t.loc[k, 'Normal|VaR 2.5%'], ci_lo=ci[0], ci_hi=ci[1])
    return out


def b5_components(w=(0.25, 0.25, 0.25, 0.25)):
    W = np.array(w)
    lr = joint_returns(['SPY', 'TLT', 'GLD', 'BTC'], start='2014-09-18')
    R = 100 * (np.exp(lr) - 1)
    S = np.cov(R.values.T)
    sp = np.sqrt(W @ S @ W)
    z = stats.norm.ppf(1 - 0.01)
    mvar = z * (S @ W) / sp
    cvar = W * mvar
    Lp = -(R.values @ W)
    v, e = hs_var_es(Lp, 0.025)
    ces = -(R.values[Lp >= v] * W).mean(axis=0)
    return dict(N=len(R), var_p=z * sp, mvar=mvar, cvar=cvar, share=cvar / (z * sp), es_p=e, ces=ces,
                ces_share=ces / e, undiv=(z * np.sqrt(np.diag(S)) * W).sum())


def b6_es_boot():
    out = {}
    for k in ['sp500', 'bet']:
        L = (-100 * log_returns(k)).values
        for tag, x in [('full', L), ('w500', L[-500:])]:
            es = hs_var_es(x, 0.025)[1]
            ci_iid, _ = boot_ci(x, lambda y: hs_var_es(y, 0.025)[1])
            ci_blk, _ = boot_ci(x, lambda y: hs_var_es(y, 0.025)[1], block=20)
            out[f'{k}_{tag}'] = dict(N=len(x), es=es, iid_lo=ci_iid[0], iid_hi=ci_iid[1], blk_lo=ci_blk[0],
                                     blk_hi=ci_blk[1])
    return out


def c1_capital(position=1_000_000):
    lr_b = joint_returns(BVB, start=C_START)
    lr_b = lr_b[(lr_b != 0).any(axis=1)]                # fara zilele in care nicio actiune nu s-a schimbat
    start = str(lr_b.index[0].date())
    Rb = 100 * (np.exp(lr_b) - 1)
    rb = np.log(1 + Rb.mean(axis=1) / 100) * 100          # portofoliu cu ponderi egale, rebalansat zilnic
    rs = 100 * joint_returns(['SPY']).loc[start:]['SPY']
    out = dict(start=start, stocks=BVB)
    series = {'bvb': rb, 'spy': rs}
    for k, r in series.items():
        L = -r
        v1, e1 = hs_var_es(L, 0.025)
        L10 = -r.rolling(10).sum().dropna()
        e10 = hs_var_es(L10, 0.025)[1]
        # perioada de stres: fereastra de 250 de zile cu cel mai mare ES 2,5%
        roll = pd.Series([hs_var_es(L.iloc[i - 250:i], 0.025)[1] for i in range(250, len(L) + 1)],
                         index=L.index[249:])
        es_stress = roll.max()
        end_s = roll.idxmax()
        start_s = L.index[L.index.get_loc(end_s) - 249]
        params, mu, sig = garch_filter(r)
        z = ((r - mu) / sig).dropna().values
        m1, s1 = garch_next(r, params)
        nz = -z
        q = np.quantile(nz, 1 - 0.025)
        fhs1 = -m1 + s1 * nz[nz >= q].mean()
        sim = fhs_mc(r, params, z, 10, n_paths=100_000, seed=SEED)
        fhs10 = hs_var_es(sim, 0.025)[1]
        out[k] = dict(N=len(r), sd=r.std(), es1=e1, var1=v1, es10_sqrt=np.sqrt(10) * e1, es10_hs=e10,
                      es_stress1=es_stress, es_stress10=np.sqrt(10) * es_stress, stress_from=str(start_s.date()),
                      stress_to=str(end_s.date()), fhs1=fhs1, fhs10=fhs10, sigma_now=s1,
                      cap_hs=np.sqrt(10) * e1 / 100 * position, cap_stress=np.sqrt(10) * es_stress / 100 * position,
                      cap_stress_lh20=np.sqrt(20) * es_stress / 100 * position,
                      cap_fhs=fhs10 / 100 * position, zero_share=float((Rb == 0).mean().mean()) if k == 'bvb' else 0.0)
    out['corr_bvb_spy'] = float(pd.concat([rb, rs], axis=1, join='inner').corr().iloc[0, 1])
    # grafic
    fig, ax = plt.subplots(figsize=(8.5, 3.3))
    labs = ['HS, sqrt(10) x 1-day', 'HS, 10-day sums', 'Stressed 250 days, sqrt(10)', 'FHS Monte Carlo, 10-day']
    keys = ['es10_sqrt', 'es10_hs', 'es_stress10', 'fhs10']
    xx = np.arange(len(labs))
    vb = [out['bvb'][c] for c in keys]
    vs = [out['spy'][c] for c in keys]
    ax.bar(xx - 0.2, vb, 0.38, color=IDAred, label='BVB blue chips, equal weights')
    ax.bar(xx + 0.2, vs, 0.38, color=MainBlue, label='S&P 500 (SPY)')
    for i in range(len(labs)):
        ax.text(xx[i] - 0.2, vb[i] + 0.2, f'{vb[i]:.1f}', ha='center', fontsize=7)
        ax.text(xx[i] + 0.2, vs[i] + 0.2, f'{vs[i]:.1f}', ha='center', fontsize=7)
    ax.set_xticks(xx)
    ax.set_xticklabels(labs, fontsize=8)
    ax.set_ylabel('10-day ES 2.5% (% of position)')
    ax.set_title(f'Capital per unit invested, {start[:4]}-2026')
    legend_outside_bottom(ax, 2, -0.14)
    save_fig('ch7_sem_capital')
    return out

## Run

In [ ]:
for f in [a1_normal, a2_student, a3_bond, a4_bonds, a5_components, a6_cf]:
    print(f.__name__, f())
print('B1', b1_bet_hs())
print('B2', b2_bet_gpd())
print('B3', b3_btc_fhs())
print(pd.DataFrame(b4_cf_vs_empirical()).round(2))
print('B5', b5_components())
print(pd.DataFrame(b6_es_boot()).round(2))
print('C1', c1_capital())

![ch7_sem_bet_hs](./ch7_sem_bet_hs.png)

Output: `ch7_sem_bet_hs.pdf`

![ch7_sem_bet_gpd](./ch7_sem_bet_gpd.png)

Output: `ch7_sem_bet_gpd.pdf`

![ch7_sem_btc_fhs](./ch7_sem_btc_fhs.png)

Output: `ch7_sem_btc_fhs.pdf`

![ch7_sem_capital](./ch7_sem_capital.png)

Output: `ch7_sem_capital.pdf`